
# Phase 2 — Time Series Feature Engineering + Machine Learning  
## Beginner-friendly, practical, and engaging

**Course:** Data Science with Python  
**Module:** Time Series Analysis  
**Phase 2 Focus:** Feature engineering and machine learning  
**Suggested duration:** 6 hours  

---

## Why this phase matters

In Phase 1, students learned how to:
- work with dates
- visualize time series
- think about trend, seasonality, and stationarity

Now we move to a very practical question:

> How do we turn time series data into features that machine learning models can use?

This is one of the most useful real-world skills in time series work.

---

## Learning goals

By the end of this notebook, students should be able to:

- explain why forecasting with machine learning needs feature engineering
- create time-based features such as month, day of week, and quarter
- create lag features
- create rolling statistics
- understand why lag features are powerful
- avoid data leakage in time series modeling
- split data chronologically into train and test sets
- build simple forecasting baselines
- train a machine learning regression model for forecasting
- compare a baseline forecast to a machine learning model
- evaluate forecasts using MAE, RMSE, and MAPE
- interpret forecasting results in practical terms

---

## Suggested teaching structure for a 6-hour session

### Hour 1
- Review of Phase 1
- Why time series + ML needs feature engineering
- Load dataset and inspect target

### Hour 2
- Calendar features
- Lag features
- Rolling features
- Visual intuition

### Hour 3
- Data leakage in time series
- Chronological train/test split
- Forecasting baseline models

### Hour 4
- Build first ML forecasting model
- Train, predict, evaluate

### Hour 5
- Add more features
- Compare models
- Interpret results

### Hour 6
- Forecasting metrics
- Reflection and exercises
- Practical lessons learned


In [ ]:

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline



# Part 1: Load the daily retail dataset

We will use a practical retail-style dataset.

Columns include:
- `sales` (target we want to forecast)
- `orders`
- `website_visits`
- `ad_spend`
- `promo`
- `holiday_season`

**File used:** `time_series_phase2_retail_daily.csv`


In [ ]:

# Load the dataset
df = pd.read_csv("/mnt/data/time_series_phase2_retail_daily.csv")

# Show first rows
df.head()


In [ ]:

# Convert date to datetime and set as index
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

df.head()


In [ ]:

# Basic shape and summary
print("Rows and columns:", df.shape)
print()
print(df.dtypes)


In [ ]:

# Plot target series
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["sales"])
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Daily Sales")
plt.show()



## Key idea for this phase

A machine learning model does not automatically “understand time.”

If you give the model only a row number and a target, it usually misses:
- seasonality
- recency
- short-term momentum
- repeating weekly patterns

So we need to create useful features from time.



# Part 2: Calendar features

Calendar features are often the first and easiest time series features to build.

Examples:
- year
- month
- quarter
- day of week
- weekend flag

These can help the model learn repeating time patterns.


In [ ]:

# Create calendar features
df_fe = df.copy()

df_fe["year"] = df_fe.index.year
df_fe["month"] = df_fe.index.month
df_fe["quarter"] = df_fe.index.quarter
df_fe["day_of_week"] = df_fe.index.dayofweek
df_fe["day_of_month"] = df_fe.index.day
df_fe["is_weekend"] = (df_fe["day_of_week"] >= 5).astype(int)

df_fe[[
    "sales", "year", "month", "quarter", "day_of_week", "day_of_month", "is_weekend"
]].head()


In [ ]:

# Average sales by day of week
dow_avg = df_fe.groupby("day_of_week")["sales"].mean()

plt.figure(figsize=(7, 4))
plt.bar(dow_avg.index.astype(str), dow_avg.values)
plt.xlabel("Day of Week (0=Mon, 6=Sun)")
plt.ylabel("Average Sales")
plt.title("Average Sales by Day of Week")
plt.show()



## Teaching point

Calendar features are simple, but often surprisingly powerful.

They help the model learn patterns like:
- Friday and Saturday are stronger
- some months are busier
- holiday season matters



# Part 3: Lag features

Lag features are among the most important features in time series machine learning.

A lag feature means:

- yesterday's sales
- sales 7 days ago
- sales 30 days ago

These help the model learn from the past.

For example:
- `lag_1` means yesterday's value
- `lag_7` means the value from one week ago


In [ ]:

# Create lag features for sales
df_fe["sales_lag_1"] = df_fe["sales"].shift(1)
df_fe["sales_lag_7"] = df_fe["sales"].shift(7)
df_fe["sales_lag_14"] = df_fe["sales"].shift(14)
df_fe["sales_lag_30"] = df_fe["sales"].shift(30)

df_fe[[
    "sales", "sales_lag_1", "sales_lag_7", "sales_lag_14", "sales_lag_30"
]].head(12)



## Why lag features are so powerful

Time series values are often related to recent past values.

Examples:
- today's sales may be related to yesterday's sales
- this Saturday may resemble last Saturday
- this month may look like the same time last month

Lag features give the model a memory of the past.


In [ ]:

# Visual compare: actual sales vs lag 7
sample = df_fe[["sales", "sales_lag_7"]].dropna().iloc[:60]

plt.figure(figsize=(12, 4))
plt.plot(sample.index, sample["sales"], label="Sales")
plt.plot(sample.index, sample["sales_lag_7"], label="Lag 7", linestyle="--")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Sales vs 7-Day Lag")
plt.legend()
plt.show()



# Part 4: Rolling features

Rolling features summarize recent history.

Examples:
- 7-day average sales
- 14-day average sales
- 7-day standard deviation

These help the model capture:
- local trend
- recent level
- recent volatility


In [ ]:

# Create rolling features
df_fe["sales_roll_mean_7"] = df_fe["sales"].shift(1).rolling(7).mean()
df_fe["sales_roll_mean_14"] = df_fe["sales"].shift(1).rolling(14).mean()
df_fe["sales_roll_std_7"] = df_fe["sales"].shift(1).rolling(7).std()

df_fe[[
    "sales", "sales_roll_mean_7", "sales_roll_mean_14", "sales_roll_std_7"
]].head(20)


In [ ]:

# Plot sales and 7-day rolling mean
plot_df = df_fe[["sales", "sales_roll_mean_7"]].dropna().iloc[:90]

plt.figure(figsize=(12, 4))
plt.plot(plot_df.index, plot_df["sales"], alpha=0.5, label="Sales")
plt.plot(plot_df.index, plot_df["sales_roll_mean_7"], label="7-Day Rolling Mean")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Sales with Rolling Mean")
plt.legend()
plt.show()



## Very important detail: why use `.shift(1)` before rolling?

We use:

```python
df["sales"].shift(1).rolling(7).mean()
```

instead of:

```python
df["sales"].rolling(7).mean()
```

because we want the feature to use **only past data**.

If we include today's value inside today's feature, that is leakage.



# Part 5: Data leakage in time series

This is one of the most important practical ideas in forecasting.

**Leakage** means the model gets information from the future, directly or indirectly.

Examples:
- using future values in feature construction
- random train/test split
- scaling or fitting transforms using future periods

Leakage makes model performance look unrealistically good.


In [ ]:

# Bad example (for explanation only): this rolling mean leaks today's sales into today's feature
df_fe["bad_roll_mean_7"] = df_fe["sales"].rolling(7).mean()

df_fe[["sales", "sales_roll_mean_7", "bad_roll_mean_7"]].head(12)



## Teaching point

The bad feature is not always obvious to beginners.

That is why forecasting requires discipline:
- use only the past
- never let the target leak into features for the same timestamp



# Part 6: Drop rows with missing lag/rolling values

Lag and rolling features naturally create missing values at the beginning.

That is normal.

We usually remove those rows before modeling.


In [ ]:

# Drop rows with missing values created by lag/rolling features
model_df = df_fe.dropna().copy()

print("Original rows:", len(df_fe))
print("Rows after dropping missing lag/rolling values:", len(model_df))
model_df.head()



# Part 7: Forecasting baselines

Before training a machine learning model, we should build a simple baseline.

This is a very important real-world habit.

Examples of simple baselines:
- yesterday's value
- value from 7 days ago
- rolling average

If a fancy model cannot beat a simple baseline, then the fancy model is not very useful.


In [ ]:

# Create baseline predictions
model_df["baseline_lag_1"] = model_df["sales_lag_1"]
model_df["baseline_lag_7"] = model_df["sales_lag_7"]
model_df["baseline_roll_7"] = model_df["sales_roll_mean_7"]

model_df[["sales", "baseline_lag_1", "baseline_lag_7", "baseline_roll_7"]].head()



# Part 8: Chronological train/test split

In time series forecasting, the split must respect time.

We train on earlier dates and test on later dates.

This simulates real forecasting.


In [ ]:

# Chronological split: train on earlier data, test on later data
train = model_df.loc[: "2024-06-30"].copy()
test = model_df.loc["2024-07-01":].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Train end  :", train.index.max())
print("Test start :", test.index.min())



## Why not random split?

Because forecasting is about predicting the future.

A random split mixes time periods and makes the task unrealistically easy.



# Part 9: Forecast evaluation metrics

We will use three practical metrics:

### 1. MAE
Mean Absolute Error  
Average absolute forecast error

### 2. RMSE
Root Mean Squared Error  
Punishes large errors more strongly

### 3. MAPE
Mean Absolute Percentage Error  
Useful when business users like percentage interpretation


In [ ]:

# Helper function for forecast metrics
def forecast_report(y_true, y_pred, label="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"--- {label} ---")
    print(f"MAE : {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MAPE: {mape:.2f}%")
    print()


In [ ]:

# Evaluate baseline forecasts
forecast_report(test["sales"], test["baseline_lag_1"], label="Baseline: Lag 1")
forecast_report(test["sales"], test["baseline_lag_7"], label="Baseline: Lag 7")
forecast_report(test["sales"], test["baseline_roll_7"], label="Baseline: Rolling Mean 7")



## Important lesson

Simple baselines can already be strong.

Do not assume machine learning will automatically do better.



# Part 10: Build a first machine learning forecasting model

We will start with a simple regression model.

Machine learning forecasting usually means:
- create features from the past and the calendar
- use those features to predict the target

This is still a regression problem, because we predict a number.


In [ ]:

# Choose features for the first forecasting model
feature_cols = [
    "year", "month", "quarter", "day_of_week", "day_of_month", "is_weekend",
    "sales_lag_1", "sales_lag_7", "sales_lag_14", "sales_lag_30",
    "sales_roll_mean_7", "sales_roll_mean_14", "sales_roll_std_7",
    "orders", "website_visits", "ad_spend", "promo", "holiday_season"
]

X_train = train[feature_cols]
y_train = train["sales"]

X_test = test[feature_cols]
y_test = test["sales"]

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)


In [ ]:

# Train a simple Linear Regression forecasting model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on test period
lr_pred = lr_model.predict(X_test)

forecast_report(y_test, lr_pred, label="Linear Regression Forecast")


In [ ]:

# Plot actual vs predicted for the test period
plt.figure(figsize=(12, 4))
plt.plot(y_test.index, y_test.values, label="Actual Sales")
plt.plot(y_test.index, lr_pred, label="Linear Regression Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Actual vs Linear Regression Forecast")
plt.legend()
plt.show()



## What students should notice

Ask:
- Does the model follow the overall movement?
- Is it smoother or noisier than actual sales?
- Does it seem better than the baselines?



# Part 11: Compare baseline vs machine learning model


In [ ]:

# Put all results into one table
results = pd.DataFrame([
    {
        "model": "Baseline Lag 1",
        "MAE": mean_absolute_error(y_test, test["baseline_lag_1"]),
        "RMSE": mean_squared_error(y_test, test["baseline_lag_1"]) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - test["baseline_lag_1"]) / y_test)) * 100
    },
    {
        "model": "Baseline Lag 7",
        "MAE": mean_absolute_error(y_test, test["baseline_lag_7"]),
        "RMSE": mean_squared_error(y_test, test["baseline_lag_7"]) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - test["baseline_lag_7"]) / y_test)) * 100
    },
    {
        "model": "Baseline Rolling Mean 7",
        "MAE": mean_absolute_error(y_test, test["baseline_roll_7"]),
        "RMSE": mean_squared_error(y_test, test["baseline_roll_7"]) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - test["baseline_roll_7"]) / y_test)) * 100
    },
    {
        "model": "Linear Regression",
        "MAE": mean_absolute_error(y_test, lr_pred),
        "RMSE": mean_squared_error(y_test, lr_pred) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - lr_pred) / y_test)) * 100
    }
])

results.sort_values("RMSE")



## This is a key real-world habit

Always compare your ML model to a baseline.

A model is only useful if it improves on something simple and defensible.



# Part 12: Try a second machine learning model

To show that different models behave differently, we will try a Random Forest regressor.

We are not tuning it deeply here.  
The goal is just to compare behavior.


In [ ]:

# Train a Random Forest forecasting model
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

forecast_report(y_test, rf_pred, label="Random Forest Forecast")


In [ ]:

# Compare all models
results_2 = pd.DataFrame([
    {
        "model": "Baseline Lag 1",
        "MAE": mean_absolute_error(y_test, test["baseline_lag_1"]),
        "RMSE": mean_squared_error(y_test, test["baseline_lag_1"]) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - test["baseline_lag_1"]) / y_test)) * 100
    },
    {
        "model": "Baseline Lag 7",
        "MAE": mean_absolute_error(y_test, test["baseline_lag_7"]),
        "RMSE": mean_squared_error(y_test, test["baseline_lag_7"]) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - test["baseline_lag_7"]) / y_test)) * 100
    },
    {
        "model": "Baseline Rolling Mean 7",
        "MAE": mean_absolute_error(y_test, test["baseline_roll_7"]),
        "RMSE": mean_squared_error(y_test, test["baseline_roll_7"]) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - test["baseline_roll_7"]) / y_test)) * 100
    },
    {
        "model": "Linear Regression",
        "MAE": mean_absolute_error(y_test, lr_pred),
        "RMSE": mean_squared_error(y_test, lr_pred) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - lr_pred) / y_test)) * 100
    },
    {
        "model": "Random Forest",
        "MAE": mean_absolute_error(y_test, rf_pred),
        "RMSE": mean_squared_error(y_test, rf_pred) ** 0.5,
        "MAPE": np.mean(np.abs((y_test - rf_pred) / y_test)) * 100
    }
])

results_2.sort_values("RMSE")


In [ ]:

# Plot actual vs multiple forecasts
plt.figure(figsize=(12, 4))
plt.plot(y_test.index, y_test.values, label="Actual Sales")
plt.plot(y_test.index, test["baseline_lag_7"].values, label="Baseline Lag 7", alpha=0.8)
plt.plot(y_test.index, lr_pred, label="Linear Regression", alpha=0.9)
plt.plot(y_test.index, rf_pred, label="Random Forest", alpha=0.9)
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Actual vs Forecasts")
plt.legend()
plt.show()



## Teaching discussion

Ask students:
- Which forecast seems to track the shape best?
- Which model seems smoother?
- Which model seems to react to spikes better?
- Is the best model also the most interpretable?



# Part 13: Feature importance intuition

Linear Regression gives coefficients.  
Random Forest gives feature importances.

These are not exactly the same idea, but both can help us understand what matters.


In [ ]:

# Linear Regression coefficients
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lr_model.coef_
}).sort_values("coefficient", ascending=False)

coef_df.head(10)


In [ ]:

# Random Forest feature importances
rf_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

rf_importance.head(10)



## What students should learn here

In forecasting, recent history often matters a lot.

Commonly strong features:
- lag_1
- lag_7
- rolling mean
- day of week
- promo or holiday variables



# Part 14: Practical lessons about ML for forecasting

### 1. Feature engineering is central
Without good features, the model cannot use time well.

### 2. Baselines matter
Always compare to something simple.

### 3. Leakage is dangerous
One leaky feature can make the model look much better than it really is.

### 4. Chronological split is essential
Forecasting is about the future, not random holdout rows.

### 5. Different models behave differently
Simple models can work surprisingly well.



# Part 15: Exercises

## Exercise 1
Create `sales_lag_2` and `sales_lag_21`.

## Exercise 2
Create a 30-day rolling mean feature.

## Exercise 3
Train a Linear Regression model using only:
- calendar features
- lag_1
- lag_7
- rolling mean 7

Compare it with the full model.

## Exercise 4
Try a different train/test split date.

## Exercise 5
Explain in plain English:
- lag feature
- rolling feature
- baseline forecast
- data leakage
- chronological split

## Exercise 6
Which feature do you think is most useful for forecasting this series, and why?


In [ ]:

# Starter code for Exercise 1
model_df["sales_lag_2"] = model_df["sales"].shift(2)
model_df["sales_lag_21"] = model_df["sales"].shift(21)

model_df[["sales", "sales_lag_2", "sales_lag_21"]].head(25)


In [ ]:

# Starter code for Exercise 3
small_feature_cols = [
    "year", "month", "quarter", "day_of_week", "day_of_month", "is_weekend",
    "sales_lag_1", "sales_lag_7", "sales_roll_mean_7"
]

small_train = train[small_feature_cols]
small_test = test[small_feature_cols]

small_model = LinearRegression()
small_model.fit(small_train, y_train)
small_pred = small_model.predict(small_test)

forecast_report(y_test, small_pred, label="Smaller Linear Regression Model")



# Part 16: Key takeaways

- Time series machine learning depends heavily on **feature engineering**
- Calendar features help models learn repeating patterns
- Lag features give the model memory of the past
- Rolling features summarize recent history
- Use `.shift()` carefully to avoid leakage
- Forecasting baselines are essential
- Train/test split must respect time order
- Machine learning forecasts should be compared to simple alternatives
- Good forecasting is often more about feature design and evaluation than about fancy models



## End of Phase 2

A natural Phase 3 would cover:
- classical forecasting models
- ARIMA intuition
- forecasting with trend and seasonality
- one-step and multi-step forecasting
